In [1]:
import pandas as pd

from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data = pd.read_csv("Grocery_Dataset.csv")
data.head()

,TransactionID,CustomerID,Date,Time,Store,Item,Category,Quantity,UnitPrice,TotalAmount,PaymentMethod
0,TXN100001,CUST00314,2025-03-16,20:54:14,Riverside Branch,Butter,Dairy,1,123.36,123.36,UPI
1,TXN100001,CUST00314,2025-03-16,20:54:14,Riverside Branch,Bread,Bakery,1,40.33,40.33,UPI
2,TXN100001,CUST00314,2025-03-16,20:54:14,Riverside Branch,Pasta,Grocery,2,81.05,162.10,UPI
3,TXN100001,CUST00314,2025-03-16,20:54:14,Riverside Branch,Biscuits,Snacks,2,26.43,52.86,UPI
4,TXN100001,CUST00314,2025-03-16,20:54:14,Riverside Branch,Milk,Dairy,2,53.01,106.02,UPI


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55189 entries, 0 to 55188
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TransactionID  55189 non-null  object 
 1   CustomerID     55189 non-null  object 
 2   Date           55189 non-null  object 
 3   Time           55189 non-null  object 
 4   Store          55189 non-null  object 
 5   Item           55189 non-null  object 
 6   Category       55189 non-null  object 
 7   Quantity       55189 non-null  int64  
 8   UnitPrice      55189 non-null  float64
 9   TotalAmount    55189 non-null  float64
 10  PaymentMethod  55189 non-null  object 
dtypes: float64(2), int64(1), object(8)
memory usage: 4.6+ MB


In [4]:
data.isnull().sum()

TransactionID    0
CustomerID       0
Date             0
Time             0
Store            0
Item             0
Category         0
Quantity         0
UnitPrice        0
TotalAmount      0
PaymentMethod    0
dtype: int64

In [5]:
basket = data.groupby(["TransactionID", "Item"])["Quantity"].sum().unstack(fill_value = 0)

In [6]:
basket = basket > 0
display(basket.head())

Item,Apples,Bananas,Biscuits,Bread,Butter,Cereal,Cheese,ChickenBreast,Chocolate,Coffee,...,Onions,OrangeJuice,Pasta,Potatoes,Rice,Sugar,Tea,TomatoSauce,Tomatoes,Yogurt
TransactionID,,,,,,,,,,,,,,,,,,,,,
TXN100001,False,False,True,True,True,False,False,False,False,False,...,False,False,True,False,False,False,True,True,False,False
TXN100002,True,False,False,True,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
TXN100003,False,False,False,False,True,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
TXN100004,False,False,True,True,True,False,True,True,False,True,...,True,False,True,False,False,False,False,False,False,False
TXN100005,False,False,False,True,False,False,False,False,True,False,...,False,False,True,True,False,False,False,False,False,False


In [7]:
frequent_itemsets = apriori(
    basket,
    min_support = 0.4,
    use_colnames = True
)

In [8]:
frequent_itemsets = frequent_itemsets.sort_values(
    by ='support', ascending = False
)

In [9]:
display(frequent_itemsets)

,support,itemsets
1,0.654824,(Butter)
0,0.651176,(Bread)
3,0.649294,(Milk)
2,0.647294,(Eggs)
4,0.532353,"(Bread, Butter)"
6,0.530941,"(Bread, Milk)"
8,0.528706,"(Milk, Butter)"
7,0.528353,"(Eggs, Butter)"
5,0.527882,"(Bread, Eggs)"
9,0.527412,"(Milk, Eggs)"


In [10]:
rules = association_rules(
    frequent_itemsets,
    num_itemsets=len(basket),
    metric = "confidence",
    min_threshold = 0.60
)

In [11]:
rules = rules.sort_values(by="confidence", ascending = False)

In [13]:
rules = rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
]

In [14]:
display(rules)

,antecedents,consequents,support,confidence,lift
14,"(Milk, Butter)",(Bread),0.453059,0.856920,1.315957
26,"(Milk, Eggs)",(Bread),0.451882,0.856792,1.315761
18,"(Bread, Eggs)",(Butter),0.452118,0.856474,1.307947
25,"(Bread, Eggs)",(Milk),0.451882,0.856029,1.318399
30,"(Milk, Eggs)",(Butter),0.451412,0.855900,1.307070
20,"(Eggs, Butter)",(Bread),0.452118,0.855711,1.314101
32,"(Eggs, Butter)",(Milk),0.451412,0.854375,1.315853
31,"(Milk, Butter)",(Eggs),0.451412,0.853805,1.319037
12,"(Bread, Milk)",(Butter),0.453059,0.853313,1.303118
24,"(Bread, Milk)",(Eggs),0.451882,0.851097,1.314853
